# Deploying Iris-detection model using Vertex AI


### Install Vertex AI SDK for Python and other required packages



In [1]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud Project Information

In [2]:
PROJECT_ID = "iitmbs-mlops"
LOCATION = "us-central1"

### Set GCS Information

In [3]:
BUCKET_URI = f"gs://iitmbs-mlops-21f1000344"

### Initialize Vertex AI SDK for Python

In [4]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [5]:
import os
import sys

### Setup Git Repository

In [6]:
! git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /home/jupyter/.git/


In [7]:
! git config --global user.email "chandrakarsatvik@gmail.com"

In [8]:
!git config --global user.name "Satvik Chandrakar"

### Install & Configure DVC

In [9]:
! pip3 install dvc dvc-gs

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 65.1 MB/s  0:00:00
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144591 sha256=bde3915e7803f68af1b18c349378a9a93bfde7d46d426a86aaf16234451a3f20
  Stored in directory: /home/jupyter/.cache/pip/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built antlr4

In [10]:
! dvc init

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/iterative/dvc>


In [11]:
! git add .dvc

### Configure GCS as Remote Storage

In [12]:
! dvc remote add -d myremote {BUCKET_URI}

Setting 'myremote' as a default remote.


In [13]:
! dvc remote modify myremote credentialpath iitmbs-mlops-0be300234da6.json

In [14]:
! git add .dvc/config

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics
from sklearn.metrics import accuracy_score

### Load Data

In [16]:
data = pd.read_csv('data/iris.csv')
data.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [17]:
data.shape

(150, 5)

### Track Data with DVC

In [18]:
! dvc add data/iris.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding data/iris.csv to cache         0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/data/iris.c0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 15.76file/s]

To track the changes with git, run:

	git add data/.gitignore data/iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [19]:
! git add data/.gitignore data/iris.csv.dvc

In [20]:
! dvc push

Pushing
!
  0% Checking cache in 'iitmbs-mlops-21f1000344/files/md5'| |0/? [00:00<?,    ?f
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Pushing to gs                         0/1 [00:00<?,     ?file/s]

!

  0%|          |/home/jupyter/.dvc/cache/files/0.00/3.77k [00:00<?,        ?B/s]

100%|██████████|/home/jupyter/.dvc/cache/f3.77k/3.77k [00:00<00:00,    32.7kB/s]

                                                                                
100%|██████████|Pushing to gs                     1/1 [00:00<00:00,  5.54file/s]
Pushing                                                                         
1 file pushed


### Train Model

In [21]:
train, test = train_test_split(data, test_size = 0.2, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [22]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.967


In [23]:
# Save model locally
import joblib

joblib.dump(mod_dt, f"artifacts/model.joblib")

['artifacts/model.joblib']

### Track Model with DVC

In [24]:
! dvc add artifacts/model.joblib

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding artifacts/model.joblib to cache0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/artifacts/m0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 23.82file/s]

To track the changes with git, run:

	git add artifacts/model.joblib.dvc artifacts/.gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [25]:
! git add artifacts/model.joblib.dvc artifacts/.gitignore

In [26]:
! dvc push

Pushing
!
  0% Checking cache in 'iitmbs-mlops-21f1000344/files/md5'| |0/? [00:00<?,    ?f
 50% Querying cache in 'iitmbs-mlops-21f1000344/files/md5'|▌|1/2 [00:00<00:00,  
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Pushing to gs                         0/1 [00:00<?,     ?file/s]

!

  0%|          |/home/jupyter/.dvc/cache/files/0.00/2.50k [00:00<?,        ?B/s]

100%|██████████|/home/jupyter/.dvc/cache/f2.50k/2.50k [00:00<00:00,    21.4kB/s]

                                                                                
100%|██████████|Pushing to gs                     1/1 [00:00<00:00,  6.55file/s]
Pushing                                                                         
1 file pushed


In [27]:
! git commit -m "First iteration done with 150 rows of iris data"

[master (root-commit) 6279bcd] First iteration done with 150 rows of iris data
 7 files changed, 23 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
 create mode 100644 artifacts/.gitignore
 create mode 100644 artifacts/model.joblib.dvc
 create mode 100644 data/.gitignore
 create mode 100644 data/iris.csv.dvc


In [28]:
! git tag -a "v1.0" -m "model v1.0, 150 rows of data"

### Simulate Data Additions

In [29]:
# Simulate augmentation by duplicating with noise
augmented = data.copy()
augmented["sepal_length"] = augmented["sepal_length"] + 0.1
augmented["species"] = augmented["species"]

# Merge
data = pd.concat([data, augmented], ignore_index=True)

# Save new version
data.to_csv("data/iris.csv", index=False)

In [30]:
data.shape

(300, 5)

### Track Data Version 2 with DVC

In [31]:
! dvc add data/iris.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding data/iris.csv to cache         0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/data/iris.c0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 22.60file/s]

To track the changes with git, run:

	git add data/iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [32]:
! git add data/.gitignore data/iris.csv.dvc

In [33]:
! dvc push

Pushing
!
  0% Checking cache in 'iitmbs-mlops-21f1000344/files/md5'| |0/? [00:00<?,    ?f
 50% Querying cache in 'iitmbs-mlops-21f1000344/files/md5'|▌|1/2 [00:00<00:00,  
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Pushing to gs                         0/1 [00:00<?,     ?file/s]

!

  0%|          |/home/jupyter/.dvc/cache/files/0.00/8.27k [00:00<?,        ?B/s]

100%|██████████|/home/jupyter/.dvc/cache/f8.27k/8.27k [00:00<00:00,    69.1kB/s]

                                                                                
100%|██████████|Pushing to gs                     1/1 [00:00<00:00,  6.01file/s]
Pushing                                                                         
1 file pushed


### Train Model with Data Version 2

In [34]:
train, test = train_test_split(data, test_size = 0.2, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [35]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.933


In [36]:
# Save model locally
import joblib

joblib.dump(mod_dt, f"artifacts/model.joblib")

['artifacts/model.joblib']

### Track Model with DVC

In [37]:
! dvc add artifacts/model.joblib

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding artifacts/model.joblib to cache0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/artifacts/m0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 23.99file/s]

To track the changes with git, run:

	git add artifacts/model.joblib.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [38]:
! git add artifacts/model.joblib.dvc artifacts/.gitignore

In [39]:
! dvc push

Pushing
!
  0% Checking cache in 'iitmbs-mlops-21f1000344/files/md5'| |0/? [00:00<?,    ?f
 50% Querying cache in 'iitmbs-mlops-21f1000344/files/md5'|▌|1/2 [00:00<00:00,  
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Pushing to gs                         0/1 [00:00<?,     ?file/s]

!

  0%|          |/home/jupyter/.dvc/cache/files/0.00/2.50k [00:00<?,        ?B/s]

100%|██████████|/home/jupyter/.dvc/cache/f2.50k/2.50k [00:00<00:00,    21.5kB/s]

                                                                                
100%|██████████|Pushing to gs                     1/1 [00:00<00:00,  6.14file/s]
Pushing                                                                         
1 file pushed


In [40]:
! git commit -m "second iteration done with 300 rows of iris data"

[master 1e65e55] second iteration done with 300 rows of iris data
 2 files changed, 3 insertions(+), 3 deletions(-)


In [41]:
! git tag -a "v2.0" -m "model v2.0, 300 rows of data"

### Version Traversal with `dvc checkout`

In [49]:
!git checkout v2.0
!dvc checkout

Previous HEAD position was 6279bcd First iteration done with 150 rows of iris data
HEAD is now at 1e65e55 second iteration done with 300 rows of iris data
Building workspace index                              |4.00 [00:00, 14.7entry/s]
Comparing indexes                                    |5.00 [00:00, 1.30kentry/s]
Applying changes                                      |2.00 [00:00,   406file/s]
M       artifacts/model.joblib
M       data/iris.csv


In [50]:
data = pd.read_csv('data/iris.csv')
data.shape

(300, 5)

In [51]:
data.tail()

,sepal_length,sepal_width,petal_length,petal_width,species
295,6.8,3.0,5.2,2.3,virginica
296,6.4,2.5,5.0,1.9,virginica
297,6.6,3.0,5.2,2.0,virginica
298,6.3,3.4,5.4,2.3,virginica
299,6.0,3.0,5.1,1.8,virginica


In [52]:
train, test = train_test_split(data, test_size = 0.2, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [53]:
X_train.shape

(240, 4)

In [54]:
X_test.shape

(60, 4)

In [55]:
model = joblib.load("artifacts/model.joblib")
y_pred = model.predict(X_test)
print("Eval Accuracy:", accuracy_score(y_test, y_pred))

Eval Accuracy: 0.9333333333333333
